<a href="https://colab.research.google.com/github/Shahd799/flyrank-internship-1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shahd799/flyrank-internship-1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
----------------------------------------------------------------------

### The contract (plain words)

1. **Unit of analysis:** One row = one content page, for one client, on one
   reporting date (a page-day). This is the raw grain of
   `fact_content_daily_performance`.

2. **Table(s):** `fact_content_daily_performance`, partition `month=2026-03`.
   March is a mid-panel month, not the final month (`2026-06`), which is
   the sealed test month.

3. **Time window:** 2026-03-01 to 2026-03-31 (one calendar month).

4. **Label / proxy:** Whether a page's total impressions declined from one
   month to the next. This will be computed directly from
   `gsc_impressions` totals across two monthly partitions (e.g. March vs
   April) — not from a pre-labeled column like `trend_direction`, which is
   itself rule-derived and should never be used as a feature or a
   ready-made label (see the label-trap note from Week 2).

5. **One thing deliberately excluded:** the `ai_*` referral columns
   (`ai_chatgpt`, `ai_perplexity`, etc.) are excluded from this lane's
   features. They are mostly missing for rows without GA4 data, and this
   lane's priority question is about search visibility, not AI referral
   mix.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
)

print("Rows:", len(df))
print("Columns:", len(df.columns))
df.head()

Rows: 9841378
Columns: 31


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field classification

**Features** (knowable before the decision moment, this month's data only):
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_sessions

**Label / proxy:**
Impression decline from this month to the next month, computed from
gsc_impressions across two monthly partitions. Not yet computed in this
notebook (that happens in the modeling weeks) — this notebook only
establishes the contract and feature frame.

**Context** (for grouping/joining/splitting only, never as model input):
- report_date
- client_hash_id
- content_hash_id

**Excluded** (with reasons):
- trend_direction, trend_pct — not present in this warehouse table at all
(they belong to the Week-1 starter CSV), but noted here as a reminder:
even if a similar rule-derived column existed here, it would never be
usable as a label or feature per the Week-2 label-trap correction.
- ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta,
  ai_other — mostly missing outside GA4-available rows, not needed for
  this lane's core question.
- client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available —
  these are availability flags used for filtering, not features to learn
  from.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_frame = df[df["gsc_data_available"] == True][[
    "content_hash_id", "client_hash_id",
    "gsc_impressions", "gsc_clicks", "gsc_avg_position",
    "ga4_pageviews", "ga4_sessions"
]].copy()

print("Feature frame shape:", feature_frame.shape)
feature_frame.head()


Feature frame shape: (3611061, 7)


,content_hash_id,client_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,20,0,3.350000,NaN,NaN
1,content_05597932fe4da067,client_73cda7b4e4f265ea,1,0,0.000000,NaN,NaN
2,content_7a105f548d9c6916,client_73cda7b4e4f265ea,125,1,4.928000,NaN,NaN
3,content_905aa32a0230694e,client_73cda7b4e4f265ea,7,0,4.000000,NaN,NaN
4,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,11,0,2.272727,NaN,NaN


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*
### Query 1 — Grain check

One row should be one (report_date, client, content) combination, with no
duplicates.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Query 1: Grain probe
grain_check = (
    df.groupby(["report_date", "client_hash_id", "content_hash_id"])
    .size()
    .reset_index(name="n")
)
duplicate_rows = (grain_check["n"] > 1).sum()
print("Duplicate grain rows (should be 0):", duplicate_rows)

Duplicate grain rows (should be 0): 0


### Query 2 — Row count and date window

Confirms the slice size and that the window matches the March partition
exactly.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Query 2: Counts and window
print("Number of rows:", len(df))
print("Start date:", df["report_date"].min())
print("End date:", df["report_date"].max())

Number of rows: 9841378
Start date: 2026-03-01
End date: 2026-03-31


### Query 3 — Availability and missingness

Filters with IS TRUE on gsc_data_available (this lane depends on GSC
metrics), and checks missingness on the core features.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Query 3: Availability (IS TRUE) + missingness on core features
available = df[df["gsc_data_available"] == True]
print("Rows with GSC data available:", len(available))
print("Share of total:", round(len(available) / len(df), 3))
print()

missingness = available[
    ["gsc_impressions", "gsc_clicks", "gsc_avg_position", "ga4_pageviews", "ga4_sessions"]
].isnull().mean()
print("Missingness (share of rows) per feature, within GSC-available rows:")
print(missingness)

Rows with GSC data available: 3611061
Share of total: 0.367

Missingness (share of rows) per feature, within GSC-available rows:
gsc_impressions     0.000000
gsc_clicks          0.000000
gsc_avg_position    0.000000
ga4_pageviews       0.423246
ga4_sessions        0.423246
dtype: float64


### Five features, with "knowable at the decision moment because..."

- **gsc_impressions**: knowable because it is a completed March total —
  entirely in the past relative to any future decision.
- **gsc_clicks**: knowable for the same reason — completed March total.
- **gsc_avg_position**: knowable because it is computed from March ranking
  data only, no forward-looking information.
- **ga4_pageviews**: knowable, but only for rows where ga4_data_available
  is True — this dependency must be checked before use.
- **ga4_sessions**: same caveat as ga4_pageviews.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# The 5-feature frame, built from this month's data only
feature_frame = available[[
    "content_hash_id", "client_hash_id",
    "gsc_impressions", "gsc_clicks", "gsc_avg_position",
    "ga4_pageviews", "ga4_sessions"
]].copy()

print(feature_frame.shape)
feature_frame.head()

(3611061, 7)


,content_hash_id,client_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,20,0,3.350000,NaN,NaN
1,content_05597932fe4da067,client_73cda7b4e4f265ea,1,0,0.000000,NaN,NaN
2,content_7a105f548d9c6916,client_73cda7b4e4f265ea,125,1,4.928000,NaN,NaN
3,content_905aa32a0230694e,client_73cda7b4e4f265ea,7,0,4.000000,NaN,NaN
4,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,11,0,2.272727,NaN,NaN


### The trap: deliberately injecting a leaky feature

To confirm I understand leakage rather than just avoiding it by luck, I
build a quick proxy target for this exercise, get an honest baseline score,
then add ONE feature that the label was directly computed from — the trap
— and watch the score jump toward perfect. Then I remove it and keep the
honest number. This proxy target (quick_label) is only used for this
demonstration; it is not the real label used in later modeling weeks.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

# Quick proxy target for this exercise only: "no clicks despite impressions"
trap_df = feature_frame.dropna(subset=["gsc_avg_position", "gsc_impressions", "gsc_clicks"]).copy()
trap_df = trap_df[trap_df["gsc_impressions"] > 0]
trap_df["quick_label"] = (trap_df["gsc_clicks"] == 0).astype(int)

print("quick_label distribution:")
print(trap_df["quick_label"].value_counts())
print()

# Honest features — no relationship to how quick_label was built
honest_features = ["gsc_impressions", "gsc_avg_position"]
X = trap_df[honest_features]
y = trap_df["quick_label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
clf = DecisionTreeClassifier(max_depth=4, random_state=42)
clf.fit(X_train, y_train)
honest_score = roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])
print("Honest ROC-AUC (no leak):", honest_score)

# THE TRAP: add gsc_clicks — the exact column quick_label was computed from
leaky_features = honest_features + ["gsc_clicks"]
X_leak = trap_df[leaky_features]

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leak, y, test_size=0.2, random_state=42)
clf_leak = DecisionTreeClassifier(max_depth=4, random_state=42)
clf_leak.fit(X_train_l, y_train_l)
leaky_score = roc_auc_score(y_test_l, clf_leak.predict_proba(X_test_l)[:, 1])

print("Score WITH leaky feature (gsc_clicks, which defines quick_label):", leaky_score)
print()
print("gsc_clicks is now removed from the feature set going forward.")
print("Final honest feature set:", honest_features)

quick_label distribution:
quick_label
1    3193080
0     417981
Name: count, dtype: int64

Honest ROC-AUC (no leak): 0.8755737377766786
Score WITH leaky feature (gsc_clicks, which defines quick_label): 1.0

gsc_clicks is now removed from the feature set going forward.
Final honest feature set: ['gsc_impressions', 'gsc_avg_position']


Adding gsc_clicks — the column quick_label was directly computed from —
pushes ROC-AUC from 0.876 to 1.0, reaching perfect separation. This
confirms the leakage lesson from Week 2 on real warehouse data: a feature
derived from the same source as the label lets the model "cheat" rather
than learn a real pattern. gsc_clicks is deliberately removed, and only the
honest feature set (gsc_impressions, gsc_avg_position) is carried forward.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


- This dataset cannot prove that refreshing a page will increase traffic —
  it only shows observed historical performance (observational, not
  experimental).
- GA4 and AI referral metrics are missing for a meaningful share of rows
  (only available when ga4_data_available is True), so ga4-based features
  cannot always be used and should be flagged, not silently filled with 0.
- History depth differs by client, so a single global month window (March)
  may not be equally representative across all clients — this is a named
  limitation of this slice, not something this notebook corrects.
- Results from this data should support editorial decisions, not be
  treated as proof of what causes a page to decline or recover.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
## 4. Data limits

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.